Import Libraries

In [ ]:
import numpy as np
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

Prepare & Read Data

In [ ]:
# Your Kaggle credentials
os.environ['KAGGLE_USERNAME'] = 'malakkhaledaboahmed'
os.environ['KAGGLE_KEY'] = 'KGAT_a889cbfa4d477a03bf8001ea39431428'

In [ ]:
!kaggle datasets download -d fedesoriano/air-quality-data-set
with zipfile.ZipFile('air-quality-data-set.zip', 'r') as zip_ref:
    zip_ref.extractall('air_quality_data')
df = pd.read_csv('air_quality_data/AirQuality.csv', sep=';', decimal=',')

Exploratory Data Analysis

In [ ]:
df.head(10)
#Displays the first 10 rows to understand the data structure.
#Observation: We see -200 (missing value marker) and two empty columns at the end.

In [ ]:
#delete last 2 columns(unnamed)
df.drop(df.columns[-2:], axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
df = df.replace(-200, np.nan)
#-200 is an invalid value (air pollution can't be negative).

In [ ]:
df.info()

In [ ]:
df.describe()
#Calculates mean, std, min, max for numeric columns to understand data distribution

In [ ]:
plt.figure(figsize=(12, 6))
cols_to_check = ['PT08.S3(NOx)', 'NOx(GT)', 'PT08.S1(CO)']
sns.boxplot(data=df[cols_to_check])
plt.title("Checking for Outliers")
plt.show()
 #Many black dots outside the whiskers → outliers exist.

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(12,6))
msno.bar(df)
plt.title("Missing Values Visualization")
plt.show()
#Bar chart showing missing values. NMHC(GT) is almost completely empty

In [ ]:
df.duplicated().sum()

In [ ]:
# Merges Date and Time columns into a single datetime column for time-series analysis.
df['Timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'].astype(str).str.replace('.', ':'), errors='coerce')

#Extracts useful features (hour, day name, month) from the timestamp.
df['Hour'] = df['Timestamp'].dt.hour
df['Day_of_Week'] = df['Timestamp'].dt.day_name()
df['Month'] = df['Timestamp'].dt.month

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['Day_of_Week'] = pd.Categorical(df['Day_of_Week'], categories=days_order, ordered=True)

df['Time'] = df['Timestamp'].dt.time

print("Remaining Columns:", df.columns.tolist())
df.head()


In [ ]:
plt.figure(figsize=(12, 6))
# Line plot showing average CO concentration for each hour of the day
sns.lineplot(data=df, x='Hour', y='CO(GT)', marker='o', color='tab:blue')
plt.title('Average CO(GT) Concentration Throughout the Day (Rush Hour Analysis)')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.6)
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Concentration - CO(GT)')
plt.show()
#CO peaks during morning (8-9 AM) and evening (6-8 PM) rush hours.

In [ ]:
# Create a figure with two subplots side-by-side
fig, ax = plt.subplots(1, 2, figsize=(20, 6))

# Plot 1: Average Nitrogen Oxides (NOx) by Day of the Week (Bar Chart)
sns.barplot(data=df, x='Day_of_Week', y='NOx(GT)', ax=ax[0], palette='coolwarm')
ax[0].set_title('Average NOx(GT) Concentration by Day of the Week')
ax[0].set_xlabel('Day of the Week')
ax[0].set_ylabel('NOx(GT) Mean')
ax[0].tick_params(axis='x', rotation=45)

# Plot 2: Seasonal Trend of Benzene (C6H6) across Months (Line Plot)
sns.lineplot(data=df, x='Month', y='C6H6(GT)', marker='s', ax=ax[1], color='red')
ax[1].set_title('Seasonal Trend of Benzene (C6H6) Concentration')
ax[1].set_xlabel('Month')
ax[1].set_ylabel('C6H6(GT) Mean')
ax[1].set_xticks(range(1, 13))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()
#NOx is higher on weekdays, lower on weekends.
#Benzene is higher in winter months (Nov-Feb), lower in summer.

## **Visulization before Preprocessing**

In [ ]:
plt.figure(figsize=(10, 5))
msno.matrix(df)
plt.title('Missing Values Pattern (Before Cleaning)')
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(df['CO(GT)'], bins=50, color='red')
plt.title('CO(GT) Distribution with -200 values (Raw Data)')
plt.xlabel('Value (Notice the spike at -200)')
plt.show()

plt.figure(figsize=(10, 6))
plt.scatter(df['CO(GT)'], df['NMHC(GT)'], alpha=0.5, color='orange')
plt.title('Raw Data: CO vs NMHC (Notice the clusters at -200)')
plt.xlabel('CO(GT)')
plt.ylabel('NMHC(GT)')
plt.grid(True)
plt.show()

# هيت ماب بتوضح هل فيه علاقة بين فقدان البيانات في الأعمدة المختلفة
msno.heatmap(df.replace(-200, np.nan))
plt.title('Correlation of Missing Values between Sensors')
plt.show()

## **Preprocessing**

Dealing With Duplicates

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()

Dealing with Nulls

In [ ]:
df = df.drop('NMHC(GT)', axis=1)
df = df.drop('Date', axis=1)
df = df.drop('Time', axis=1)

In [ ]:
#ffill() fills missing values with the previous row's value. bfill() fills remaining ones with the next row's value. Good for time-series data.
df = df.ffill().bfill()

In [ ]:
plt.figure(figsize=(12,6))
msno.bar(df)
plt.title("Missing Values After Processing")
plt.show()
#All bars are solid dark color → no missing values remain.


In [ ]:
df_categorical = df.select_dtypes(include=['object'])
df_numerical = df.select_dtypes(exclude=['object'])

In [ ]:
print("categorical columns:")
print(df_categorical.columns)
print("numerical columns:")
print(df_numerical.columns)


Dealing with outliers

In [ ]:
# Define the columns to check
cols_to_check = ['PT08.S3(NOx)', 'NOx(GT)', 'PT08.S1(CO)']

# Calculate the number of outliers for each column
for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers_count = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"Column {col}: Number of Outliers = {outliers_count}")

In [ ]:
# Check outliers in NOx(GT) and inspect their dates
Q1 = df['NOx(GT)'].quantile(0.25)
Q3 = df['NOx(GT)'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

nox_outliers = df[(df['NOx(GT)'] < lower) | (df['NOx(GT)'] > upper)]

print("First 10 outliers in NOx(GT):")
print(nox_outliers[['Timestamp','NOx(GT)', 'PT08.S3(NOx)']].head(10))

print("\nAre these values on the same day (2004-11-03)?")
problem_date = pd.to_datetime('2004-11-03').date()
same_day = nox_outliers[nox_outliers['Timestamp'].dt.date == problem_date]
print(f"On 2004-11-03: {len(same_day)} Outlier(s)")
print(f"On other days: {len(nox_outliers) - len(same_day)} Outlier(s)")

In [ ]:
# Check outliers in PT08.S3(NOx)
Q1 = df['PT08.S3(NOx)'].quantile(0.25)
Q3 = df['PT08.S3(NOx)'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['PT08.S3(NOx)'] < lower_bound) | (df['PT08.S3(NOx)'] > upper_bound)]
print(f"Number of Outliers: {len(outliers)}")
print(outliers[['PT08.S3(NOx)', 'Timestamp', 'CO(GT)']].head(10))

In [ ]:
# Adjust display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 25)

# Check the entire suspicious day
suspicious_day = df[df['Timestamp'].dt.date == pd.to_datetime('2004-11-03').date()]

print("="*80)
print("Analysis of Suspicious Day: 2004-11-03")
print("="*80)
print(f"Number of rows on this day: {len(suspicious_day)} rows")
print("-"*80)

print("\nData (first 20 rows):")
print("-"*80)
print(suspicious_day[['Timestamp', 'PT08.S3(NOx)', 'CO(GT)', 'NOx(GT)']].head(20).to_string(index=False))
print("-"*80)

print("\nStatistics of Suspicious Day:")
print("-"*80)
print(suspicious_day[['PT08.S3(NOx)', 'CO(GT)', 'NOx(GT)']].describe().to_string())
print("="*80)

In [ ]:
# Compare 2004-11-03 with a normal day e.g., 2004-12-03
normal_day = df[df['Timestamp'].dt.date == pd.to_datetime('2004-12-03').date()]
print("Normal day (2004-12-03):")
print(normal_day[['PT08.S3(NOx)', 'CO(GT)', 'NOx(GT)']].describe())

In [ ]:
# Delete only the hours with anomalous values (00:00 to 07:00)
df = df[~((df['Timestamp'].dt.date == pd.to_datetime('2004-11-03').date()) &
          (df['Timestamp'].dt.hour <= 7))]

print(f"Number of rows after deletion: {len(df)}")

In [ ]:
# Example for CO(GT) column
Q1 = df['CO(GT)'].quantile(0.05)   # Lower bound (1st percentile)
Q99 = df['CO(GT)'].quantile(0.95)  # Upper bound (99th percentile)

# Clip the values
df['CO(GT)'] = df['CO(GT)'].clip(lower=Q1, upper=Q99)

In [ ]:
# 1. Calculate bounds (1% and 99%) as you did for CO
Q1 = df['PT08.S5(O3)'].quantile(0.01)
Q99 = df['PT08.S5(O3)'].quantile(0.99)

print(f"Lower bound (1%): {Q1}")
print(f"Upper bound (99%): {Q99}")

# 2. Apply clipping
df['PT08.S5(O3)'] = df['PT08.S5(O3)'].clip(lower=Q1, upper=Q99)

# 3. Verify the result
print("\nStatistics after processing:")
print(df['PT08.S5(O3)'].describe())

## **Visulization after Preprocessing**

In [ ]:
# Create a correlation heatmap to identify highly related features (Multicollinearity).
# This helps in Feature Selection by identifying sensors that provide redundant information.
plt.figure(figsize=(12, 8))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap (Post-Cleaning)')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.histplot(df['CO(GT)'].dropna(), kde=True, ax=axes[0,0], color='blue').set_title('CO(GT) Distribution')
sns.histplot(df['C6H6(GT)'].dropna(), kde=True, ax=axes[0,1], color='green').set_title('Benzene (C6H6) Distribution')
sns.histplot(df['T'].dropna(), kde=True, ax=axes[1,0], color='orange').set_title('Temperature Distribution')
sns.histplot(df['RH'].dropna(), kde=True, ax=axes[1,1], color='red').set_title('Humidity Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Convert 'Timestamp' to datetime objects and extract the hour (0-23)
# This helps in identifying peak pollution periods throughout the day.
df['Hour'] = pd.to_datetime(df['Timestamp'], format='%H.%M.%S').dt.hour
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Hour', y='PT08.S1(CO)', estimator='mean')
plt.title('Average CO Sensor Response by Hour of Day')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.show()

## **Feature Engineering**

In [ ]:
# --- 1. Time-based and human activity features ---
df['Hour'] = df['Timestamp'].dt.hour

# Rush hour feature (7–9 AM and 5–7 PM)
df['Is_RushHour'] = df['Hour'].apply(lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0)

# Working day feature (0 for weekends, 1 for weekdays)
df['Is_WorkingDay'] = df['Timestamp'].dt.dayofweek.apply(lambda x: 0 if x >= 5 else 1)

# Add season feature
if 'Season' not in df.columns:
    def get_season(month):
        if month in [12, 1, 2]: return 'Winter'
        elif month in [3, 4, 5]: return 'Spring'
        elif month in [6, 7, 8]: return 'Summer'
        else: return 'Autumn'
    df['Season'] = df['Month'].apply(get_season)

# Convert seasons into dummy variables
season_dummies = pd.get_dummies(df['Season'], prefix='Season').astype(int)
df = pd.concat([df, season_dummies], axis=1)

# --- 2. Lag and rolling statistical features ---
df['CO_lag1'] = df['CO(GT)'].shift(1)
df['T_lag1'] = df['T'].shift(1)
df['NOx_lag1'] = df['NOx(GT)'].shift(1)
df['CO_mean_3h'] = df['CO(GT)'].rolling(window=3).mean()
df['S1_std_6h'] = df['PT08.S1(CO)'].rolling(window=6).std()
df['Temp_Diff'] = df['T'].diff().fillna(0)

# --- 3. Additional features (atmospheric interactions) ---
df['T_RH_Interaction'] = df['T'] * df['RH']

In [ ]:
# Select sensor columns and initialize scaler
sensor_cols = ['PT08.S1(CO)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'NO2(GT)']
scaler = StandardScaler()

# Apply scaling to the sensor data
sensors_scaled = scaler.fit_transform(df[sensor_cols])

# Perform PCA
pca = PCA(n_components=2)
sensors_pca = pca.fit_transform(sensors_scaled)

# Add PCA components to dataframe
df['Sensor_PCA1'] = sensors_pca[:, 0]
df['Sensor_PCA2'] = sensors_pca[:, 1]

In [ ]:
df['CO_Trend'] = df['CO(GT)'].diff()

def get_day_period(h):
    if 6 <= h <= 10: return 1
    if 16 <= h <= 20: return 2
    if 21 <= h or h <= 5: return 3
    return 0

df['Day_Period'] = df['Hour'].apply(get_day_period)
df['CO_Volatility_3h'] = df['CO(GT)'].rolling(window=3).std()
df['hour_sin'] = np.sin(2 * np.pi * df['Hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour']/24)

print("--- [Step 1] Manual Feature Engineering Output ---")
print(f"Total columns in main df: {df.shape[1]}")
display(df.head())

In [ ]:
# --- FINAL FEATURE SELECTION & REFINEMENT FOR CLUSTERING ---

# Define the most informative features for Clustering
# We prioritize temporal behavior, pollution trends, PCA sensor fusion,
# volatility, and atmospheric interactions

final_feature_columns = [

    'CO_mean_3h',        # Rolling mean of CO
    'CO_Trend',          # Hourly CO change
    'CO_Volatility_3h',  # Pollution fluctuation
    'CO_lag1',           # Previous hour memory

    'Sensor_PCA1',       # Main sensor behavior pattern
    'Sensor_PCA2',       # Secondary sensor pattern

    'T_RH_Interaction',  # Temperature-Humidity interaction

    'PT08.S2(NMHC)',     # Hydrocarbon sensor
    'C6H6(GT)',          # Benzene level
    'PT08.S4(NO2)',      # NO2 sensor
    'NOx(GT)'            # Nitrogen Oxides
]

# Create final cleaned dataset for clustering
# Drop rows with NaN values caused by lag / rolling calculations

df_final_clean = df.dropna(subset=final_feature_columns)

# Final feature matrix
X_final_v3 = df_final_clean[final_feature_columns]

print("Dataset successfully prepared for Clustering.")
print(f"Number of Features: {X_final_v3.shape[1]}")
print(f"Dataset Shape: {X_final_v3.shape}")
print("Final Features:")
print(list(X_final_v3.columns))
display(X_final_v3.head())

In [ ]:
# Select features (use the names defined in Feature Engineering)
features_to_scale = [col for col in final_feature_columns if col not in ['hour_sin', 'hour_cos']]
circular_features = ['hour_sin', 'hour_cos']

#  Apply scaling only to non-circular features using RobustScaler
robust_scaler = RobustScaler()
scaled_data = robust_scaler.fit_transform(df_final_clean[features_to_scale])

#  Combine scaled features with sin and cos (without scaling)
X_scaled_part = pd.DataFrame(scaled_data, columns=features_to_scale, index=df_final_clean.index)
X_final = pd.concat([X_scaled_part, df_final_clean[circular_features]], axis=1)

print("Scale Done (Excluding Sin/Cos)! Ready for Elbow Method.")

Model Devolopment

In [ ]:
#  Elbow Method
inertia = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_final)
    inertia.append(kmeans.inertia_)

# 5. Plot the results
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertia, marker='o', linestyle='--', color='b')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Air Quality Clustering')
plt.xticks(K_range)
plt.grid(True)
plt.show()

In [ ]:
#  Apply the model with 3 clusters
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(X_final)

# Add the results to the original dataset
df_final_clean['Cluster'] = clusters

print("Clustering Complete! Each row is now assigned to a group (0, 1, or 2).")

# Check the number of points in each cluster
print("\nPoints per Cluster:")
print(df_final_clean['Cluster'].value_counts())

In [ ]:
# Analyze clusters based on average values
analysis = df_final_clean.groupby('Cluster').agg({
    'CO(GT)': 'mean',
    'NOx(GT)': 'mean',
    'T': 'mean',
    'Hour': 'mean',
    'Is_RushHour': 'mean' # Shows the proportion of rush hour occurrence in each cluster
}).sort_values(by='CO(GT)') # Sort by carbon monoxide level

print("Cluster Analysis Table")
display(analysis)

In [ ]:
#  calc Silhouette Score
score = silhouette_score(X_final, clusters)

print(f"Silhouette Score: {score:.3f}")

In [ ]:
#  Define a consistent min_samples value ---
target_min_samples = 22

#  Plot K-Distance based on the new min_samples ---
nn = NearestNeighbors(n_neighbors=target_min_samples)
nn.fit(X_final)
distances, _ = nn.kneighbors(X_final)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.axhline(y=1.8, color='r', linestyle='--', label='Suggested Start EPS')
plt.axhline(y=2.2, color='g', linestyle='--', label='Suggested End EPS')
plt.title(f"K-Distance Graph (n_neighbors={target_min_samples})")
plt.legend()
plt.show()

#  Improve the loop to search for the best eps ---
eps_range = np.arange(1.0, 2.6, 0.1)
scores = []
noise_ratios = []

for eps in eps_range:
    db = DBSCAN(eps=eps, min_samples=target_min_samples).fit(X_final)
    labels = db.labels_

    mask = labels != -1
    n_clusters = len(set(labels[mask]))

    if n_clusters > 1:
        # Calculate the score using only non-noise points
        score = silhouette_score(X_final[mask], labels[mask])
        scores.append(score)
    else:
        scores.append(-1)

    noise_ratios.append(np.sum(labels == -1) / len(labels))

# Select the best eps
optimal_eps = eps_range[np.argmax(scores)]
print(f"Optimal eps found: {optimal_eps:.2f} with Silhouette: {max(scores):.4f}")

In [ ]:
# Apply the final model
best_dbscan = DBSCAN(eps=optimal_eps, min_samples=target_min_samples)
db_labels = best_dbscan.fit_predict(X_final)
df_final_clean['DBSCAN_Cluster'] = db_labels

#  Analyze the results
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
print(f"Number of clusters found: {n_clusters}")
print(f"Noise Percentage: {np.sum(db_labels == -1)/len(db_labels)*100:.2f}%")

# Plot for verification
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_final_clean,
    x='Sensor_PCA1', y='Sensor_PCA2',
    hue='DBSCAN_Cluster', palette='bright',
    style=(df_final_clean['DBSCAN_Cluster'] == -1),
    markers={True: 'X', False: 'o'}, s=50
)
plt.title(f'Final DBSCAN Improvement (eps={optimal_eps:.2f})')
plt.show()

# Cluster averages
display(df_final_clean.groupby('DBSCAN_Cluster')[['CO(GT)', 'NOx(GT)', 'T', 'Is_RushHour']].mean())